In [299]:
import torch
import pandas as pd
from transformers import BertConfig
from geomstats.geometry.torus import Torus
from schedule import LinearBetaSchedule
from sde_lib import DiffusionMixture
from solver import get_pc_sampler
from foldingdiff.bert_for_diffusion import BertForDiffusion
from foldingdiff.angles_and_coords import create_new_chain_nerf
from Bio.PDB import PDBParser
import nglview as nv
import torch.nn.functional as F

In [300]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
max_seq_len = 128
min_seq_len = 40
ft_names = ["phi", "psi", "omega", "tau", "CA:C:1N", "C:1N:1CA"]

In [301]:
angles_per_residue = 6
cfg = BertConfig(
    max_position_embeddings=max_seq_len,
    num_attention_heads=6,
    hidden_size=192,
    intermediate_size=384,
    num_hidden_layers=6,
    position_embedding_type="relative_key",
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
    use_cache=False,
    _attn_implementation="eager"
)

modelf = BertForDiffusion(config=cfg, ft_names=ft_names).to(device)
modelf.load_state_dict(torch.load('./forward_bert.pt', weights_only=True))

<All keys matched successfully>

In [302]:
modelb = BertForDiffusion(config=cfg, ft_names=ft_names).to(device)
modelb.load_state_dict(torch.load('./backward_bert.pt', weights_only=True))

<All keys matched successfully>

In [303]:
beta_schedule = LinearBetaSchedule(beta_0=0.2, beta_f=0.001, t0=0, tf=1.0)   
mix = DiffusionMixture(
    beta_schedule,
    mix_type="log",
    drift_scale=1.0,
    pred=False,
    pred_scale=1.0,
    prior_type="unif",
)

In [ ]:
fdrift_fn = mix.get_drift_fn(modelf)
bdrift_fn = mix.rev().get_drift_fn(modelb)
sde = mix.approx(fdrift_fn, bdrift_fn, True)
samples = []
for n_residues in range(min_seq_len, max_seq_len + 1):
    manifold = Torus((n_residues - 1) * 6)
    shape = (1,)
    sampler = get_pc_sampler(manifold=manifold, sde=sde, shape=shape, N=500, eps=0.001, device=device)
    sample = sampler(prior_samples=None)

    angles_rec = torch.atan2(sample[:, 1::2], sample[:, 0::2])
    angles_rec = F.pad(angles_rec, (1, angles_per_residue - 1), "constant", 0)
    angles_rec = angles_rec.reshape(shape[0], -1, angles_per_residue)

    df_rec = pd.DataFrame(angles_rec[0].cpu().numpy(), columns=ft_names)
    rec_pdb = create_new_chain_nerf(f'rec_prot_{n_residues}.pdb', df_rec)
    samples.append(angles_rec)

In [431]:
def plot_protein(pdb_file):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("prot", pdb_file)
    view = nv.show_biopython(structure)
    return view

In [432]:
plot_protein(rec_pdb)

NGLWidget()